Load cortical and subcortical atlases, extract specific ROIs, and combine with new numbering (starting with 1, per [mrtrix3 requirements](https://mrtrix.readthedocs.io/en/latest/quantitative_structural_connectivity/structural_connectome.html#preparing-a-parcellation-image-for-connectome-generation)—see [labelconvert](https://mrtrix.readthedocs.io/en/latest/quantitative_structural_connectivity/labelconvert_tutorial.html#labelconvert-tutorial) for more details)

In [2]:
import os
import ants
import pandas as pd
import numpy as np
import nibabel as nib

from glob import glob
from nilearn.image import resample_to_img, binarize_img

In [3]:
tian_scale = 'S2'

In [4]:
ref_dir = os.path.abspath('/Users/dsj3886/data_local/reference')
print(ref_dir)

tian_fpath = os.path.join(ref_dir, f'Tian_Subcortex_{tian_scale}_7T.nii')
tian_img = nib.load(tian_fpath)
carpet_fpath = os.path.join(ref_dir,'tpl-MNI152NLin2009cAsym_space-MNI_res-01_label-carpet_atlas.nii')
carpet_img = nib.load(carpet_fpath)

/Users/dsj3886/data_local/reference


In [5]:
tian_s3_dict = {
             'PUT-VA-lh': 42,
             'PUT-DA-lh': 43,
             'PUT-VP-lh': 44,
             'PUT-DP-lh': 45,
             'CAU-VA-lh': 46,
             'CAU-DA-lh': 47,
             'pCAU-lh': 54,
             'PUT-VA-rh': 15,
             'PUT-DA-rh': 16,
             'PUT-VP-rh': 17,
             'PUT-DP-rh': 18,
             'CAU-VA-rh': 19,
             'CAU-DA-rh': 20,
             'pCAU-rh': 27,
            }

tian_s2_dict = {
            'aPUT-lh': 31,
            'pPUT-lh': 32,
            'aCAU-lh': 33,
            'pCAU-lh': 34,
            'aPUT-rh': 14,
            'pPUT-rh': 15,
            'aCAU-rh': 16,
            'pCAU-rh': 17,
            }

carpet_dict = {'L-HG': 189, 
               'L-PP': 187, 'L-PT': 191, 
               'L-STGa': 117, 'L-STGp': 119,
               'R-HG': 190, 
               'R-PP': 188, 'R-PT': 192, 
               'R-STGa': 118, 'R-STGp': 120, }

vis_carpet_dict = {'L-LOCsup'   : 143,
                   'L-LOCinf'   : 145,
                   'L-IntraCalc': 147,
                   'L-TempOccFus': 177,
                   'L-OccFus'   : 178,
                   'L-SupraCalc': 193,
                   'L-OccPole'  : 195,
                   'R-LOCsup'   : 144,
                   'R-LOCinf'   : 146,
                   'R-IntraCalc': 148,
                   'R-TempOccFus': 178,
                   'R-OccFus'   : 179,
                   'R-SupraCalc': 194,
                   'R-OccPole'  : 196,
                    }

In [6]:
def generate_mask(atlas_img, labelnum, labelname, new_labelnum=None):    
    atlas_data = atlas_img.get_fdata()
    atlas_affine = atlas_img.affine
    
    mask_data = np.zeros((atlas_data.shape))
    if new_labelnum:
        mask_data[np.where(atlas_data == labelnum)] = new_labelnum
    else:
        mask_data[np.where(atlas_data == labelnum)] = labelnum

    mask_img = nib.Nifti1Image(mask_data, atlas_affine)

    return mask_img

Extract ROIs

In [7]:
#atlas_base = 'atlas-custom_subcort-tians3_cort-carpet'
#atlas_base = 'atlas-custom_subcort-tians3_cort-vis-carpet'
atlas_base = f'atlas-custom_subcort-tian{tian_scale}_cort-aud-vis-carpet'
print('atlas_base =', atlas_base)
out_dir = os.path.join('/Users/dsj3886/data_local/derivatives', atlas_base)
os.makedirs(out_dir, exist_ok=True)
print(out_dir)
new_lut = []
new_labelnum = 1

atlas_base = atlas-custom_subcort-tianS2_cort-aud-vis-carpet
/Users/dsj3886/data_local/derivatives/atlas-custom_subcort-tianS2_cort-aud-vis-carpet


In [8]:
# Tian atlas
if tian_scale == 'S2':
    tian_dict = tian_s2_dict
elif tian_scale == 'S3':
    tian_dict = tian_s3_dict

for rx, region_label in enumerate(tian_dict.keys()):
    print(region_label)
    labelnum = tian_dict[region_label]

    mask_orig_img = generate_mask(tian_img, labelnum, region_label, new_labelnum)
    mask_img = resample_to_img(mask_orig_img, carpet_img, interpolation='nearest')

    out_base = f'{atlas_base}_{region_label}.nii.gz'
    out_fpath = os.path.join(out_dir, out_base)
    nib.save(mask_img, out_fpath)

    new_lut.append([new_labelnum, region_label])

    new_labelnum +=1

aPUT-lh
pPUT-lh
aCAU-lh
pCAU-lh
aPUT-rh
pPUT-rh
aCAU-rh
pCAU-rh


In [9]:
# Carpet aseg atlas with auditory ROIs
for rx, region_label in enumerate(carpet_dict.keys()):
    print(region_label)
    labelnum = carpet_dict[region_label]

    mask_img = generate_mask(carpet_img, labelnum, region_label, new_labelnum)

    out_base = f'{atlas_base}_{region_label}.nii.gz'
    out_fpath = os.path.join(out_dir, out_base)
    nib.save(mask_img, out_fpath)

    new_lut.append([new_labelnum, region_label])

    new_labelnum +=1


L-HG
L-PP
L-PT
L-STGa
L-STGp
R-HG
R-PP
R-PT
R-STGa
R-STGp


In [10]:
# Carpet aseg atlas with visual ROIs
for rx, region_label in enumerate(vis_carpet_dict.keys()):
    print(region_label)
    labelnum = vis_carpet_dict[region_label]

    mask_img = generate_mask(carpet_img, labelnum, region_label, new_labelnum)

    out_base = f'{atlas_base}_{region_label}.nii.gz'
    out_fpath = os.path.join(out_dir, out_base)
    nib.save(mask_img, out_fpath)

    new_lut.append([new_labelnum, region_label])

    new_labelnum +=1


L-LOCsup
L-LOCinf
L-IntraCalc
L-TempOccFus
L-OccFus
L-SupraCalc
L-OccPole
R-LOCsup
R-LOCinf
R-IntraCalc
R-TempOccFus
R-OccFus
R-SupraCalc
R-OccPole


Combine ROIs into new atlas

In [11]:
# check new LUT
print(new_lut)

[[1, 'aPUT-lh'], [2, 'pPUT-lh'], [3, 'aCAU-lh'], [4, 'pCAU-lh'], [5, 'aPUT-rh'], [6, 'pPUT-rh'], [7, 'aCAU-rh'], [8, 'pCAU-rh'], [9, 'L-HG'], [10, 'L-PP'], [11, 'L-PT'], [12, 'L-STGa'], [13, 'L-STGp'], [14, 'R-HG'], [15, 'R-PP'], [16, 'R-PT'], [17, 'R-STGa'], [18, 'R-STGp'], [19, 'L-LOCsup'], [20, 'L-LOCinf'], [21, 'L-IntraCalc'], [22, 'L-TempOccFus'], [23, 'L-OccFus'], [24, 'L-SupraCalc'], [25, 'L-OccPole'], [26, 'R-LOCsup'], [27, 'R-LOCinf'], [28, 'R-IntraCalc'], [29, 'R-TempOccFus'], [30, 'R-OccFus'], [31, 'R-SupraCalc'], [32, 'R-OccPole']]


In [12]:
## COPY FROM FLT CODE
new_lut_df = pd.DataFrame(new_lut)
print(new_lut_df)

     0             1
0    1       aPUT-lh
1    2       pPUT-lh
2    3       aCAU-lh
3    4       pCAU-lh
4    5       aPUT-rh
5    6       pPUT-rh
6    7       aCAU-rh
7    8       pCAU-rh
8    9          L-HG
9   10          L-PP
10  11          L-PT
11  12        L-STGa
12  13        L-STGp
13  14          R-HG
14  15          R-PP
15  16          R-PT
16  17        R-STGa
17  18        R-STGp
18  19      L-LOCsup
19  20      L-LOCinf
20  21   L-IntraCalc
21  22  L-TempOccFus
22  23      L-OccFus
23  24   L-SupraCalc
24  25     L-OccPole
25  26      R-LOCsup
26  27      R-LOCinf
27  28   R-IntraCalc
28  29  R-TempOccFus
29  30      R-OccFus
30  31   R-SupraCalc
31  32     R-OccPole


In [15]:
# save new LUT
new_lut_fpath = os.path.join(out_dir, atlas_base+'_lut.tsv')
print('saving LUT to', new_lut_fpath)

new_lut_df.to_csv(new_lut_fpath, sep='\t', header=False, index=False)

saving LUT to /Users/dsj3886/data_local/derivatives/atlas-custom_subcort-tianS2_cort-aud-vis-carpet/atlas-custom_subcort-tianS2_cort-aud-vis-carpet_lut.tsv


In [13]:
atlas_base

'atlas-custom_subcort-tianS2_cort-aud-vis-carpet'

### Create an atlas with the correct LUT values (**updated for Revision-2**)

In [ ]:
import nibabel as nib
from nilearn import image
import numpy as np
import os

# Load reference image shape from first LUT entry
ref_img = image.load_img(os.path.join(out_dir, f"{atlas_base}_{new_lut[0][1]}.nii.gz"))
combined = np.zeros(ref_img.shape[:3], dtype=np.int16)

# Iterate over LUT in order to assign correct label numbers
for labelnum, region_label in new_lut:
    fpath = os.path.join(out_dir, f"{atlas_base}_{region_label}.nii.gz")
    img = image.load_img(fpath)
    mask = img.get_fdata() > 0
    combined[mask] = labelnum

# Make new NIfTI
atlas_img = image.new_img_like(ref_img, combined)
atlas_img.set_data_dtype(np.int16)

atlas_joined_fpath = os.path.join(out_dir, atlas_base + "_atlas.nii.gz")
nib.save(atlas_img, atlas_joined_fpath)

In [16]:
atlas_joined_fpath

'/Users/dsj3886/data_local/derivatives/atlas-custom_subcort-tianS2_cort-aud-vis-carpet/atlas-custom_subcort-tianS2_cort-aud-vis-carpet_atlas.nii.gz'